In [1]:
%%configure -f
{"vCores": 16, "defaultLakehouse": {"name": "<lakehouse_name>", "id": "<lakehouse_id>", "workspaceId": "<workspace_id>"}}

# Harrier on RagBench TechQA

Harrier retrieval workflow on an enterprise-style benchmark from RagBench.

This notebook does three things:

1. Loads the **RagBench `techqa`** subset (enterprise technical support documentation)
2. Benchmarks **Harrier 270M** and **Harrier 0.6B** as dense retrievers
3. Benchmarks **Harrier 0.6B + BM25 blending**, tunes **alpha**, and ends with a **three-way comparison table**

About `harrier`

`harrier-oss-v1` is a family of multilingual text embedding models developed by Microsoft. The models use decoder-only architectures with last-token pooling and L2 normalization to produce dense text embeddings. They can be applied to a wide range of tasks, including but not limited to retrieval, clustering, semantic similarity, classification, bitext mining, and reranking. The models achieve state-of-the-art results on the Multilingual MTEB v2 benchmark as of the release date.

27b model is too large to use in a Fabric notebook, without GPUs but the 270M and 0.6B models are perfect, especially the 270M model despite being small is extremely capable and beats many other models in the MTEB2 benchmark (including the text-embedding-ada-002 model)

More : https://huggingface.co/microsoft/harrier-oss-v1-27b



About `techqa` Dataset

- https://huggingface.co/datasets/galileo-ai/ragbench
- It is enterprise-oriented, not biomedical or trivia-heavy
- Each example has **5 candidate documents**, which makes retrieval tradeoffs visible
- It is a better fit for Harrier + BM25 blending than the 2-3 document finance/manual subsets

The evaluation uses RagBench's sentence-level relevance annotations to derive **gold relevant document ids** per question.


In [2]:
import importlib
import os
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=4.49,<5.0",
        "accelerate",
        "huggingface-hub",
        "datasets",
        "rank_bm25",
        "pandas",
        "tqdm",
    ]
)

for key in list(sys.modules.keys()):
    if key in ("torch", "transformers") or key.startswith("torch.") or key.startswith("transformers."):
        del sys.modules[key]
importlib.invalidate_caches()

import json
import os
import random
import re
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import display
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

torch.set_num_threads(min(16, os.cpu_count() or 16))

print(f"torch {torch.__version__}")
print(f"CPUs visible: {os.cpu_count()}")


torch 2.11.0+cu130
CPUs visible: 16


In [3]:
DATASET_CONFIG = "techqa"
SPLIT = "test"
SAMPLE_SIZE = 80  # set to None to evaluate every labeled example
TOP_K = 3
DOC_MAX_TOKENS = 4096
QUERY_MAX_TOKENS = 1024
DOC_BATCH_SIZE = 2
ALPHAS = [round(i / 10, 1) for i in range(11)]
SEED = 42
LAKEHOUSE_DIR = "/lakehouse/default/Files"

def extract_doc_ids(sentence_keys):
    doc_ids = set()
    for key in sentence_keys or []:
        match = re.match(r"^(\d+)", str(key))
        if match:
            doc_ids.add(int(match.group(1)))
    return doc_ids

raw_dataset = load_dataset("galileo-ai/ragbench", DATASET_CONFIG, split=SPLIT)

valid_examples = []
for row in raw_dataset:
    relevant_doc_ids = extract_doc_ids(row.get("all_relevant_sentence_keys"))
    utilized_doc_ids = extract_doc_ids(row.get("all_utilized_sentence_keys"))
    gold_doc_ids = sorted(relevant_doc_ids or utilized_doc_ids)
    if not gold_doc_ids:
        continue

    documents = [doc.replace("\x00", " ").strip() for doc in row["documents"]]
    valid_examples.append(
        {
            "id": row["id"],
            "question": row["question"].strip(),
            "response": row["response"],
            "documents": documents,
            "gold_doc_ids": gold_doc_ids,
            "dataset_name": row.get("dataset_name", DATASET_CONFIG),
        }
    )

rng = random.Random(SEED)
if SAMPLE_SIZE is None or SAMPLE_SIZE >= len(valid_examples):
    examples = list(valid_examples)
else:
    examples = sorted(rng.sample(valid_examples, SAMPLE_SIZE), key=lambda row: row["id"])

doc_counts = [len(example["documents"]) for example in examples]
doc_char_lengths = [len(doc) for example in examples for doc in example["documents"]]

print(f"Loaded {len(raw_dataset)} raw rows from RagBench/{DATASET_CONFIG}:{SPLIT}")
print(f"Kept {len(valid_examples)} rows with usable relevance annotations")
print(f"Evaluating {len(examples)} rows")
print(f"Average candidate docs/example: {np.mean(doc_counts):.1f}")
print(f"Average document length: {np.mean(doc_char_lengths):,.0f} chars")
print(f"Alpha sweep: {ALPHAS}")

pd.DataFrame(
    [
        {
            "id": example["id"],
            "question": example["question"][:110],
            "gold_doc_ids": example["gold_doc_ids"],
            "doc_count": len(example["documents"]),
        }
        for example in examples[:5]
    ]
)


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/5.40M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/5.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1192 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/304 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/314 [00:00<?, ? examples/s]

Loaded 314 raw rows from RagBench/techqa:test
Kept 260 rows with usable relevance annotations
Evaluating 80 rows
Average candidate docs/example: 5.0
Average document length: 3,740 chars
Alpha sweep: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


,id,question,gold_doc_ids,doc_count
0,techqa_DEV_Q008,How can I export a private key from DataPower ...,[2],5
1,techqa_DEV_Q012,I receive the following during UDX compilation...,[1],5
2,techqa_DEV_Q013,Additional steps to apply fix pack to Decision...,[1],5
3,techqa_DEV_Q013,Additional steps to apply fix pack to Decision...,"[2, 4]",5
4,techqa_DEV_Q014,Installing fixpacks with Installation Manager ...,[0],5


#### Adding an instruction is required for harrier model

In [4]:
QUERY_PREFIX = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: "

def last_token_pool(hidden_states, attention_mask):
    left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
    if left_padding:
        return hidden_states[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    return hidden_states[torch.arange(hidden_states.shape[0], device=hidden_states.device), sequence_lengths]

def encode_texts(model, tokenizer, texts, max_length, batch_size):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start : start + batch_size]
        batch = tokenizer(
            batch_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        with torch.no_grad():
            output = model(**batch)
        embeddings = last_token_pool(output.last_hidden_state, batch["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        vectors.append(embeddings.cpu().numpy().astype("float32"))
    return np.vstack(vectors)

def encode_query(model, tokenizer, question):
    return encode_texts(
        model,
        tokenizer,
        [QUERY_PREFIX + question],
        max_length=QUERY_MAX_TOKENS,
        batch_size=1,
    )[0]

def tokenize_bm25(text):
    return [token for token in re.findall(r"[a-z0-9]+", text.lower()) if len(token) > 1]

def normalize_scores(scores):
    scores = np.asarray(scores, dtype="float32")
    spread = scores.max() - scores.min()
    if spread <= 1e-8:
        return np.zeros_like(scores)
    return (scores - scores.min()) / spread

def top_indices(scores, top_k=TOP_K):
    return np.argsort(scores)[::-1][:top_k].tolist()

def hit_metrics(indices, gold_doc_ids):
    gold = set(gold_doc_ids)
    hit_at_1 = int(bool(indices) and indices[0] in gold)
    hit_at_3 = int(any(idx in gold for idx in indices[:3]))
    return hit_at_1, hit_at_3

def build_summary(system_name, details, doc_seconds, query_seconds, bm25_seconds=0.0):
    total = len(details)
    return {
        "system": system_name,
        "recall_at_1": round(100 * sum(row["hit_at_1"] for row in details) / total, 1),
        "recall_at_3": round(100 * sum(row["hit_at_3"] for row in details) / total, 1),
        "avg_doc_embed_ms": round(doc_seconds * 1000 / total, 1),
        "avg_query_ms": round(query_seconds * 1000 / total, 1),
        "avg_bm25_ms": round(bm25_seconds * 1000 / total, 1),
        "avg_retrieval_ms": round((query_seconds + bm25_seconds) * 1000 / total, 1),
        "doc_seconds": round(doc_seconds, 3),
        "query_seconds": round(query_seconds, 3),
        "bm25_seconds": round(bm25_seconds, 3),
        "details": details,
    }


In [5]:
MODEL_NAMES = {
    "Harrier 270M": "microsoft/harrier-oss-v1-270m",
    "Harrier 0.6B": "microsoft/harrier-oss-v1-0.6b",
}

print("Loading Harrier 270M...")
t0 = time.time()
tokenizer_270m = AutoTokenizer.from_pretrained(MODEL_NAMES["Harrier 270M"])
model_270m = AutoModel.from_pretrained(MODEL_NAMES["Harrier 270M"], torch_dtype=torch.float32).eval()
load_270m_s = time.time() - t0
print(f"  Loaded in {load_270m_s:.1f}s")

print("\nLoading Harrier 0.6B...")
t0 = time.time()
tokenizer_06b = AutoTokenizer.from_pretrained(MODEL_NAMES["Harrier 0.6B"])
model_06b = AutoModel.from_pretrained(MODEL_NAMES["Harrier 0.6B"], torch_dtype=torch.float32).eval()
load_06b_s = time.time() - t0
print(f"  Loaded in {load_06b_s:.1f}s")


Loading Harrier 270M...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

  Loaded in 6.8s


In [6]:
def evaluate_dense_model(system_name, model, tokenizer, examples, collect_hybrid_cache=False):
    details = []
    hybrid_cache = []
    doc_seconds = 0.0
    query_seconds = 0.0
    bm25_seconds = 0.0

    for example in tqdm(examples, desc=system_name):
        docs = example["documents"]

        t0 = time.time()
        doc_embeddings = encode_texts(
            model,
            tokenizer,
            docs,
            max_length=DOC_MAX_TOKENS,
            batch_size=DOC_BATCH_SIZE,
        )
        doc_seconds += time.time() - t0

        t1 = time.time()
        query_embedding = encode_query(model, tokenizer, example["question"])
        query_seconds += time.time() - t1

        dense_scores = doc_embeddings @ query_embedding
        dense_top = top_indices(dense_scores)
        hit_at_1, hit_at_3 = hit_metrics(dense_top, example["gold_doc_ids"])

        details.append(
            {
                "id": example["id"],
                "question": example["question"],
                "gold_doc_ids": example["gold_doc_ids"],
                "top_indices": dense_top,
                "top_scores": [round(float(dense_scores[idx]), 4) for idx in dense_top],
                "hit_at_1": hit_at_1,
                "hit_at_3": hit_at_3,
            }
        )

        if collect_hybrid_cache:
            t2 = time.time()
            bm25 = BM25Okapi([tokenize_bm25(doc) for doc in docs])
            bm25_scores = np.asarray(bm25.get_scores(tokenize_bm25(example["question"])), dtype="float32")
            bm25_seconds += time.time() - t2

            hybrid_cache.append(
                {
                    "id": example["id"],
                    "question": example["question"],
                    "gold_doc_ids": example["gold_doc_ids"],
                    "documents": docs,
                    "dense_scores": dense_scores.tolist(),
                    "bm25_scores": bm25_scores.tolist(),
                }
            )

    summary = build_summary(system_name, details, doc_seconds, query_seconds, bm25_seconds)
    if collect_hybrid_cache:
        summary["hybrid_cache"] = hybrid_cache
    return summary

summary_270m = evaluate_dense_model("Harrier 270M", model_270m, tokenizer_270m, examples)
summary_06b = evaluate_dense_model(
    "Harrier 0.6B",
    model_06b,
    tokenizer_06b,
    examples,
    collect_hybrid_cache=True,
)

dense_df = pd.DataFrame(
    [
        {
            "system": summary_270m["system"],
            "recall_at_1": summary_270m["recall_at_1"],
            "recall_at_3": summary_270m["recall_at_3"],
            "avg_doc_embed_ms": summary_270m["avg_doc_embed_ms"],
            "avg_retrieval_ms": summary_270m["avg_retrieval_ms"],
        },
        {
            "system": summary_06b["system"],
            "recall_at_1": summary_06b["recall_at_1"],
            "recall_at_3": summary_06b["recall_at_3"],
            "avg_doc_embed_ms": summary_06b["avg_doc_embed_ms"],
            "avg_retrieval_ms": summary_06b["avg_retrieval_ms"],
        },
    ]
)

display(dense_df)


Harrier 270M:   0%|          | 0/80 [00:00<?, ?it/s]

Harrier 0.6B:   0%|          | 0/80 [00:00<?, ?it/s]

,system,recall_at_1,recall_at_3,avg_doc_embed_ms,avg_retrieval_ms
0,Harrier 270M,75.0,93.8,5018.7,93.9
1,Harrier 0.6B,77.5,97.5,16434.1,326.8


#### Below we blend BM25 with harrier and find the right tuning parameter to improve retrieval accuracy R@3 score

In [7]:
def evaluate_hybrid_alpha(base_summary, alpha):
    details = []
    for row in base_summary["hybrid_cache"]:
        dense_scores = np.asarray(row["dense_scores"], dtype="float32")
        bm25_scores = np.asarray(row["bm25_scores"], dtype="float32")
        hybrid_scores = alpha * normalize_scores(dense_scores) + (1.0 - alpha) * normalize_scores(bm25_scores)
        hybrid_top = top_indices(hybrid_scores)
        hit_at_1, hit_at_3 = hit_metrics(hybrid_top, row["gold_doc_ids"])

        details.append(
            {
                "id": row["id"],
                "question": row["question"],
                "gold_doc_ids": row["gold_doc_ids"],
                "top_indices": hybrid_top,
                "top_scores": [round(float(hybrid_scores[idx]), 4) for idx in hybrid_top],
                "hit_at_1": hit_at_1,
                "hit_at_3": hit_at_3,
            }
        )

    summary = build_summary(
        f"Harrier 0.6B + BM25 (alpha={alpha:.1f})",
        details,
        base_summary["doc_seconds"],
        base_summary["query_seconds"],
        base_summary["bm25_seconds"],
    )
    summary["alpha"] = alpha
    return summary

alpha_summaries = [evaluate_hybrid_alpha(summary_06b, alpha) for alpha in ALPHAS]
alpha_df = pd.DataFrame(
    [
        {
            "alpha": row["alpha"],
            "recall_at_1": row["recall_at_1"],
            "recall_at_3": row["recall_at_3"],
            "avg_retrieval_ms": row["avg_retrieval_ms"],
        }
        for row in alpha_summaries
    ]
).sort_values(["recall_at_1", "recall_at_3", "alpha"], ascending=[False, False, True])

best_hybrid = sorted(
    alpha_summaries,
    key=lambda row: (row["recall_at_1"], row["recall_at_3"], -abs(row["alpha"] - 0.5)),
    reverse=True,
)[0]

print("Alpha sweep for Harrier 0.6B + BM25")
display(alpha_df)
print(f"Best alpha: {best_hybrid['alpha']:.1f}")


Alpha sweep for Harrier 0.6B + BM25


,alpha,recall_at_1,recall_at_3,avg_retrieval_ms
9,0.9,77.5,98.8,326.8
10,1.0,77.5,97.5,326.8
8,0.8,75.0,97.5,326.8
7,0.7,72.5,96.2,326.8
6,0.6,71.2,92.5,326.8
5,0.5,71.2,91.2,326.8
4,0.4,66.2,90.0,326.8
3,0.3,61.2,87.5,326.8
2,0.2,58.8,86.2,326.8
1,0.1,57.5,86.2,326.8


Best alpha: 0.9


In [8]:
comparison_df = pd.DataFrame(
    [
        {
            "system": summary_270m["system"],
            "recall_at_1": summary_270m["recall_at_1"],
            "recall_at_3": summary_270m["recall_at_3"],
            "avg_doc_embed_ms": summary_270m["avg_doc_embed_ms"],
            "avg_retrieval_ms": summary_270m["avg_retrieval_ms"],
        },
        {
            "system": summary_06b["system"],
            "recall_at_1": summary_06b["recall_at_1"],
            "recall_at_3": summary_06b["recall_at_3"],
            "avg_doc_embed_ms": summary_06b["avg_doc_embed_ms"],
            "avg_retrieval_ms": summary_06b["avg_retrieval_ms"],
        },
        {
            "system": f"Harrier 0.6B + BM25 (alpha={best_hybrid['alpha']:.1f})",
            "recall_at_1": best_hybrid["recall_at_1"],
            "recall_at_3": best_hybrid["recall_at_3"],
            "avg_doc_embed_ms": best_hybrid["avg_doc_embed_ms"],
            "avg_retrieval_ms": best_hybrid["avg_retrieval_ms"],
        },
    ]
).sort_values(["recall_at_1", "recall_at_3"], ascending=False)

print("Three-way comparison")
display(comparison_df)


Three-way comparison


,system,recall_at_1,recall_at_3,avg_doc_embed_ms,avg_retrieval_ms
2,Harrier 0.6B + BM25 (alpha=0.9),77.5,98.8,16434.1,326.8
1,Harrier 0.6B,77.5,97.5,16434.1,326.8
0,Harrier 270M,75.0,93.8,5018.7,93.9


In [9]:
interesting_indices = [
    idx
    for idx, (base_row, hybrid_row) in enumerate(zip(summary_06b["details"], best_hybrid["details"]))
    if hybrid_row["hit_at_1"] > base_row["hit_at_1"]
]

if len(interesting_indices) < 3:
    fallback = [0, min(1, len(examples) - 1), min(2, len(examples) - 1)]
    interesting_indices = list(dict.fromkeys(interesting_indices + fallback))

interesting_indices = interesting_indices[:3]

def show_example(idx):
    example = examples[idx]
    dense_270 = summary_270m["details"][idx]
    dense_06 = summary_06b["details"][idx]
    hybrid = best_hybrid["details"][idx]

    print("=" * 100)
    print(f"[{idx}] {example['question']}")
    print(f"Gold relevant docs: {example['gold_doc_ids']}")

    for label, detail in [
        ("Harrier 270M", dense_270),
        ("Harrier 0.6B", dense_06),
        (f"Harrier 0.6B + BM25 (alpha={best_hybrid['alpha']:.1f})", hybrid),
    ]:
        print(f"\n{label} | hit@1={detail['hit_at_1']} hit@3={detail['hit_at_3']}")
        for rank, doc_idx in enumerate(detail["top_indices"], start=1):
            preview = example["documents"][doc_idx].replace("\n", " ")[:220]
            score = detail["top_scores"][rank - 1]
            print(f"  #{rank} doc {doc_idx} | score={score} | {preview}...")

for idx in interesting_indices:
    show_example(idx)


[2] Additional steps to apply fix pack to Decision Center 

We are upgrading ODM 8.8.1 Decision Center to the latest 8.8.1.3.

Are there any additional steps to apply the fixpack to Decision Center after we install the fixpack with Installation Manager?
Gold relevant docs: [1]

Harrier 270M | hit@1=0 hit@3=1
  #1 doc 2 | score=0.7475 | DOWNLOADABLE FILES  ABSTRACT  This cumulative fix pack updates Operational Decision Manager V8.5.1 to V8.5.1.1.   DOWNLOAD DESCRIPTION This fix pack is entitled. Only customers who have purchased Operational Decision Man...
  #2 doc 1 | score=0.7085 | fix pack update bug correction APAR TECHNOTE (FAQ)  QUESTION  When and how should I apply a modification or fix pack to my JRules or Operational Decision Management (WODM/ODM) installation?   CAUSE The Operational Decisi...
  #3 doc 4 | score=0.6276 | FIXES ARE AVAILABLE Operational Decison Manager z/OS 8.8.1.2 Fix Pack [http://www-01.ibm.com/support/docview.wss?uid=swg24043654] Operational Decision Manager

In [10]:
results_payload = {
    "dataset": {
        "name": f"galileo-ai/ragbench::{DATASET_CONFIG}",
        "split": SPLIT,
        "evaluated_rows": len(examples),
        "filtered_rows": len(valid_examples),
        "sample_size": SAMPLE_SIZE,
        "seed": SEED,
        "top_k": TOP_K,
    },
    "systems": {
        "harrier_270m": summary_270m,
        "harrier_06b": {key: value for key, value in summary_06b.items() if key != "hybrid_cache"},
        "harrier_06b_hybrid_best": best_hybrid,
    },
    "alpha_sweep": alpha_df.sort_values("alpha").to_dict(orient="records"),
    "load_times_s": {"harrier_270m": round(load_270m_s, 1), "harrier_06b": round(load_06b_s, 1)},
}

results_path = f"{LAKEHOUSE_DIR}/harrier_ragbench_techqa_results.json"
with open(results_path, "w", encoding="utf-8") as handle:
    json.dump(results_payload, handle, indent=2)

alpha_path = f"{LAKEHOUSE_DIR}/harrier_ragbench_techqa_alpha_sweep.csv"
alpha_df.sort_values("alpha").to_csv(alpha_path, index=False)

print(f"Saved {results_path}")
print(f"Saved {alpha_path}")
print("\nBest system:")
display(comparison_df.head(1))


Saved /lakehouse/default/Files/harrier_ragbench_techqa_results.json
Saved /lakehouse/default/Files/harrier_ragbench_techqa_alpha_sweep.csv

Best system:


,system,recall_at_1,recall_at_3,avg_doc_embed_ms,avg_retrieval_ms
2,Harrier 0.6B + BM25 (alpha=0.9),77.5,98.8,16434.1,326.8


`harrier 0.6B` + BM25 alpha = 0.9 is optimal but just `harrier 270M` isnt too far behind at R@1=75% for the 80 samples